In [ ]:
!pip install netgraph

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [netgraph]


In [ ]:
import kagglehub
import pandas as pd
import numpy as np
import os

# Import useful libraries
import matplotlib.pyplot as plt
import networkx as nx
from netgraph import Graph

In [ ]:
# Download latest version
path = kagglehub.dataset_download("kannanaikkal/food-demand-forecasting")

print("Path to dataset files:", path)

Using Colab cache for faster access to the 'food-demand-forecasting' dataset.
Path to dataset files: /kaggle/input/food-demand-forecasting


In [ ]:
!ls {path}

fulfilment_center_info.csv  sample_submission.csv  train.csv
meal_info.csv		    test.csv


In [ ]:
full_path = "../kaggle/input/food-demand-forecasting"

train = pd.read_csv(os.path.join(path, "train.csv"))
test = pd.read_csv(os.path.join(path, "test.csv"))
meal_info = pd.read_csv(os.path.join(path, "meal_info.csv"))
fulfilment_center_info = pd.read_csv(os.path.join(path, "fulfilment_center_info.csv"))

# Meal info

In [ ]:
meal_info.head()

,meal_id,category,cuisine
0,1885,Beverages,Thai
1,1993,Beverages,Thai
2,2539,Beverages,Thai
3,1248,Beverages,Indian
4,2631,Beverages,Indian


In [ ]:
meal_info.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 51 entries, 0 to 50
Data columns (total 3 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   meal_id   51 non-null     int64 
 1   category  51 non-null     object
 2   cuisine   51 non-null     object
dtypes: int64(1), object(2)
memory usage: 1.3+ KB


In [ ]:
meal_info.meal_id.nunique(), meal_info.category.nunique(), meal_info.cuisine.nunique()

for col in meal_info.columns:
    unique_items = ''
    tmp=[]
    for n, item in enumerate(meal_info[col].sort_values().unique().astype(str).tolist()):
        if (n+1)%10==0:
            unique_items += ', '.join(tmp) + '\n'
            tmp = []
        else:
            tmp.append(item)
    unique_items += ', '.join(tmp) + '\n'

    print(f"{col}:\n\tUnique values ({meal_info[col].nunique()}): {unique_items}\n")

meal_id:
	Unique values (51): 1062, 1109, 1198, 1207, 1216, 1230, 1247, 1248, 1311
1445, 1525, 1543, 1558, 1571, 1727, 1754, 1770, 1778
1847, 1878, 1885, 1902, 1962, 1971, 1993, 2104, 2126
2290, 2304, 2306, 2322, 2444, 2490, 2492, 2494, 2539
2577, 2581, 2631, 2640, 2664, 2704, 2707, 2760, 2826
2956


category:
	Unique values (14): Beverages, Biryani, Desert, Extras, Fish, Other Snacks, Pasta, Pizza, Rice Bowl
Sandwich, Seafood, Soup, Starters


cuisine:
	Unique values (4): Continental, Indian, Italian, Thai




In [ ]:
meal_info.groupby(['category', 'cuisine']).count().sort_values(by='cuisine', ascending=False)

,,meal_id
category,cuisine,
Other Snacks,Thai,3
Extras,Thai,3
Soup,Thai,3
Starters,Thai,3
Beverages,Thai,3
Pasta,Italian,3
Salad,Italian,3
Sandwich,Italian,3
Beverages,Italian,3


# Fulfilment center info

In [ ]:
fulfilment_center_info.head(10)

,center_id,city_code,region_code,center_type,op_area
0,11,679,56,TYPE_A,3.7
1,13,590,56,TYPE_B,6.7
2,124,590,56,TYPE_C,4.0
3,66,648,34,TYPE_A,4.1
4,94,632,34,TYPE_C,3.6
5,64,553,77,TYPE_A,4.4
6,129,593,77,TYPE_A,3.9
7,139,693,34,TYPE_C,2.8
8,88,526,34,TYPE_A,4.1
9,143,562,77,TYPE_B,3.8


In [ ]:
fulfilment_center_info.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 77 entries, 0 to 76
Data columns (total 5 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   center_id    77 non-null     int64  
 1   city_code    77 non-null     int64  
 2   region_code  77 non-null     int64  
 3   center_type  77 non-null     object 
 4   op_area      77 non-null     float64
dtypes: float64(1), int64(3), object(1)
memory usage: 3.1+ KB


In [ ]:
for col in fulfilment_center_info.columns:
    unique_items = ''
    tmp=[]
    for n, item in enumerate(fulfilment_center_info[col].sort_values().unique().astype(str).tolist()):
        if (n+1)%10==0:
            unique_items += ', '.join(tmp) + '\n'
            tmp = []
        else:
            tmp.append(item)
    unique_items += ', '.join(tmp) + '\n'

    print(f"{col}:\n\tUnique values ({fulfilment_center_info[col].nunique()}): {unique_items}\n")

center_id:
	Unique values (77): 10, 11, 13, 14, 17, 20, 23, 24, 26
29, 30, 32, 34, 36, 39, 41, 42, 43
51, 52, 53, 55, 57, 58, 59, 61, 64
66, 67, 68, 72, 73, 74, 75, 76, 77
81, 83, 86, 88, 89, 91, 92, 93, 94
99, 101, 102, 104, 106, 108, 109, 110, 113
126, 129, 132, 137, 139, 143, 145, 146, 149
153, 157, 161, 162, 174, 177, 186


city_code:
	Unique values (51): 456, 461, 473, 478, 485, 515, 517, 522, 526
553, 556, 561, 562, 576, 577, 579, 590, 593
599, 602, 604, 609, 614, 615, 620, 628, 632
647, 648, 649, 651, 654, 658, 659, 675, 676
680, 683, 685, 693, 695, 698, 699, 700, 702
713


region_code:
	Unique values (8): 23, 34, 35, 56, 71, 77, 85, 93


center_type:
	Unique values (3): TYPE_A, TYPE_B, TYPE_C


op_area:
	Unique values (30): 0.9, 1.9, 2.0, 2.4, 2.7, 2.8, 2.9, 3.0, 3.2
3.5, 3.6, 3.7, 3.8, 3.9, 4.0, 4.1, 4.2, 4.4
4.6, 4.7, 4.8, 5.0, 5.1, 5.3, 5.6, 6.3, 6.7





# Podział na regiony i centra

Warto zwizualizować centra realizacyjne tak, żeby było widać, które ile miast obsługuje, którego typu jest, itd.

In [57]:
center_groups = fulfilment_center_info.groupby(['center_type', 'region_code', 'city_code'])['center_id'].count()

center_groups

center_type  region_code  city_code
TYPE_A       23           698          1
             34           461          1
                          526          3
                          576          1
                          602          1
                                      ..
TYPE_C       77           515          1
                          541          1
                          680          1
                          695          1
             85           699          2
Name: center_id, Length: 63, dtype: int64

In [58]:
from collections import Counter

Counter(center_groups)

Counter({1: 55, 3: 2, 2: 4, 4: 2})

# Get a glimpse of the training data

In [ ]:
train.head(10)

,id,week,center_id,meal_id,checkout_price,base_price,emailer_for_promotion,homepage_featured,num_orders
0,1379560,1,55,1885,136.83,152.29,0,0,177
1,1466964,1,55,1993,136.83,135.83,0,0,270
2,1346989,1,55,2539,134.86,135.86,0,0,189
3,1338232,1,55,2139,339.50,437.53,0,0,54
4,1448490,1,55,2631,243.50,242.50,0,0,40
5,1270037,1,55,1248,251.23,252.23,0,0,28
6,1191377,1,55,1778,183.36,184.36,0,0,190
7,1499955,1,55,1062,182.36,183.36,0,0,391
8,1025244,1,55,2707,193.06,192.06,0,0,472
9,1054194,1,55,1207,325.92,384.18,0,1,676


In [ ]:
train.groupby(['week', 'meal_id']).mean()

id  center_id  checkout_price  base_price  \
week meal_id                                                        
1    1062     1.263689e+06  83.142857      176.202727  176.202727   
     1109     1.233601e+06  83.142857      265.711948  289.463636   
     1198     1.248668e+06  65.384615      176.016154  203.690385   
     1207     1.233419e+06  82.041096      320.114658  350.067945   
     1216     1.254643e+06  83.242424      417.256364  436.736212   
...                    ...        ...             ...         ...   
145  2707     1.246571e+06  83.142857      218.343766  222.084805   
     2760     1.238502e+06  82.563380      175.040704  252.814507   
     2826     1.242649e+06  83.142857      350.731039  350.588182   
     2867     1.230558e+06  84.153846      719.729808  726.951154   
     2956     1.182009e+06  77.600000      617.094000  616.919000   

              emailer_for_promotion  homepage_featured  num_orders  
week meal_id                                                        
1    1062                  0.000000           0.000000  375.883117  
     1109                  0.467532           0.012987  830.961039  
     1198                  0.000000           0.038462  207.346154  
     1207                  0.054795           0.205479  286.205479  
     1216                  0.000000           0.181818   59.939394  
...                             ...                ...         ...  
145  2707                  0.000000           0.000000  517.311688  
     2760                  0.000000           0.352113  230.845070  
     2826                  0.000000           0.000000  333.844156  
     2867                  0.000000           0.115385   31.346154  
     2956                  0.000000           0.000000   82.475000  

[7173 rows x 7 columns]

In [ ]:
train.groupby(['week', 'center_id', 'meal_id']).mean()

id  checkout_price  base_price  \
week center_id meal_id                                          
1    10        1062     1436842.0          181.39      181.39   
               1109     1142165.0          231.86      276.48   
               1198     1421626.0          129.04      196.00   
               1207     1260561.0          322.07      382.18   
               1216     1411421.0          455.93      454.93   
...                           ...             ...         ...   
145  186       2704     1420415.0          321.13      319.13   
               2707     1013011.0          196.00      195.00   
               2760     1082117.0          182.36      261.93   
               2826     1311738.0          327.92      327.92   
               2867     1123781.0          727.53      727.53   

                        emailer_for_promotion  homepage_featured  num_orders  
week center_id meal_id                                                        
1    10        1062                       0.0                0.0       865.0  
               1109                       1.0                0.0      2672.0  
               1198                       0.0                0.0       269.0  
               1207                       0.0                0.0       769.0  
               1216                       0.0                0.0        54.0  
...                                       ...                ...         ...  
145  186       2704                       0.0                0.0        67.0  
               2707                       0.0                0.0       175.0  
               2760                       0.0                0.0        96.0  
               2826                       0.0                0.0       162.0  
               2867                       0.0                0.0        28.0  

[456548 rows x 6 columns]